# 23 — Part 4's switching results, corrected for recall of the 2020 vote

Notebook 06 defines Biden 2020 voters by recalled vote (`presvote20post`) and reports the switch rate by economy rating (31.6% among "much worse") and the gap's effect on switching (AME 3.9pp). Notebook 21 showed recall misclassifies. Here each 2024 voter in notebook 06's sample is weighted by the probability that they **reported** voting Biden in 2020, estimated in the CES 2020–24 panel by recalled 2020 status × 2024 vote × economy rating (much worse vs. the rest), with cells under 10 pooled across ratings. Every table and model of notebook 06 §7 is then re-run with weight = post weight × that probability, so recalled non-Biden voters who in fact voted Biden enter too. Intervals: 500 bootstrap resamples of the panel (the cross-section held fixed).

Fixed before estimation: notebook 06's sample and models unchanged except the weight; truth = 2020 report; panel economy item `CC24_302` (1 much better … 5 much worse).

In [1]:
import numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf, warnings
from pathlib import Path
warnings.filterwarnings('ignore'); pd.set_option('display.width', 200)
RAW = Path('../data/raw'); PROC = Path('../data/processed'); TABLES = Path('../outputs/tables')
def wmean(x, w):
    m = x.notna() & w.notna(); return np.average(x[m], weights=w[m])
a = pd.read_csv(PROC / 'ces_2024_attribution.csv', low_memory=False)
raw = pd.read_csv(RAW / 'ces' / 'ces_2024.csv', usecols=['caseid', 'presvote_prior'], low_memory=False)
a = a.merge(raw, on='caseid', how='left')
REC = {1: 'B', 2: 'T', 3: 'O', 4: 'O', 5: 'O', 6: 'NV'}
a['r'] = a.presvote_prior.map(REC); a['v'] = np.where(a.vote_R_2024 == 1, 'T', 'H')
a['k'] = np.where(a.perceived_econ == -2, 'much worse', 'rest')
X = a[a.voted_2024 & a.weight_post.notna() & a.r.notna()].copy()
print('notebook 06 sample check - recalled Biden, voted D/R:', int((X.r == 'B').sum()))
p = pd.read_csv(RAW / 'ces_panel' / 'merged_recontact_2024_vv.csv', usecols=['CC20_410', 'CC24_410', 'presvote20post_24', 'CC24_302', 'commonpostweight_24', 'commonweight_24'], low_memory=False)
p['r'] = p.presvote20post_24.map(REC); p['v'] = p.CC24_410.map({1: 'H', 2: 'T'}); p['wt'] = p.commonpostweight_24.fillna(p.commonweight_24)
p['k'] = np.where(p.CC24_302 == 5, 'much worse', 'rest'); p['tB'] = (p.CC20_410 == 1).astype(float); p['tT'] = (p.CC20_410 == 2).astype(float)
P = p.dropna(subset=['r', 'v']).copy()
def pi_table(P, col):
    out = {}
    for (r, v), g in P.groupby(['r', 'v']):
        pooled = np.average(g[col], weights=g.wt)
        for k in ['much worse', 'rest']:
            c = g[g.k == k]; out[(r, v, k)] = np.average(c[col], weights=c.wt) if len(c) >= 10 else pooled
    return out
pi = pi_table(P, 'tB')
print(pd.Series(pi).round(3).to_string())

notebook 06 sample check - recalled Biden, voted D/R: 20608
B   H  much worse    0.995
       rest          0.982
    T  much worse    0.707
       rest          0.744
NV  H  much worse    0.425
       rest          0.415
    T  much worse    0.105
       rest          0.008
O   H  much worse    0.032
       rest          0.032
    T  much worse    0.000
       rest          0.000
T   H  much worse    0.064
       rest          0.070
    T  much worse    0.000
       rest          0.001


In [2]:
def analyze(X, pi, full=True):
    X = X.copy(); X['piB'] = [pi.get((r, v, k), 0.0) for r, v, k in zip(X.r, X.v, X.k)]
    X['wc'] = X.weight_post * X.piB
    s = X[X.wc > 0]
    rates = {lab: wmean(s.loc[s.perceived_econ == val, 'vote_R_2024'], s.loc[s.perceived_econ == val, 'wc']) for val, lab in
             [(-2, 'much worse'), (-1, 'somewhat worse'), (0, 'same'), (1, 'somewhat better'), (2, 'much better')]}
    rates['all'] = wmean(s.vote_R_2024, s.wc)
    if not full: return rates, None
    w = s.wc * len(s) / s.wc.sum()
    rhs = 'real_pay_pct_chg_2020_2024_z + private_emp_pct_chg_2020_2024_z + unemp_rate_z'
    m = smf.glm(f'vote_R_2024 ~ gap + pid7c + college + age + white + {rhs} + fox + msnbc + cnn + local_tv + social', data=s, family=sm.families.Binomial(),
                freq_weights=w).fit(cov_type='cluster', cov_kwds={'groups': s.county_fips})
    ame = lambda d: np.average(m.predict(s.assign(gap=s.gap + d)) - m.predict(s), weights=w)
    return rates, {'gap_coef': m.params['gap'], 'gap_se': m.bse['gap'], 'AME_gap_minus1': ame(-1), 'AME_gap_plus1': ame(1), 'fox_coef': m.params['fox'], 'N_positive_weight': len(s)}
# published definition = recalled Biden with probability 1
pub_pi = {(r, v, k): float(r == 'B') for r in ['B', 'T', 'O', 'NV'] for v in ['H', 'T'] for k in ['much worse', 'rest']}
r0, m0 = analyze(X, pub_pi); r1, m1 = analyze(X, pi)
rng = np.random.default_rng(23); bs = []
for _ in range(500):
    Pb = P.sample(len(P), replace=True, random_state=rng.integers(1e9)); bs.append(analyze(X, pi_table(Pb, 'tB'), full=False)[0])
bs = pd.DataFrame(bs)
rates = pd.DataFrame({'recall (published)': r0, 'corrected': r1, 'lo': bs.quantile(0.025), 'hi': bs.quantile(0.975)})
print('Switch rate to Trump among Biden 2020 voters, by 2024 economy rating (weighted):'); print((100 * rates).round(1))
models = pd.DataFrame({'recall (published)': m0, 'corrected': m1}); print(); print(models.round(3))
rates.to_csv(TABLES / 'attribution_switching_recall_corrected.csv'); models.to_csv(TABLES / 'attribution_switch_model_recall_corrected.csv')

Switch rate to Trump among Biden 2020 voters, by 2024 economy rating (weighted):
                 recall (published)  corrected    lo    hi
much worse                     31.6       28.0  16.1  39.1
somewhat worse                 11.1        8.3   4.7  10.7
same                            4.6        3.4   1.9   4.5
somewhat better                 1.7        1.2   0.7   1.6
much better                     1.5        1.1   0.6   1.4
all                             6.4        5.3   3.5   7.0

                   recall (published)  corrected
gap_coef                       -0.827     -0.850
gap_se                          0.079      0.077
AME_gap_minus1                  0.039      0.034
AME_gap_plus1                  -0.025     -0.022
fox_coef                        1.868      1.864
N_positive_weight           20608.000  30339.000


In [3]:
# Mirror: recalled Trump voters switching to Harris
piT = pi_table(P, 'tT')
def mirror(X, pi):
    X = X.copy(); X['wc'] = X.weight_post * [pi.get((r, v, k), 0.0) for r, v, k in zip(X.r, X.v, X.k)]
    s = X[X.wc > 0]; return wmean(1 - s.vote_R_2024, s.wc)
pub_piT = {(r, v, k): float(r == 'T') for r in ['B', 'T', 'O', 'NV'] for v in ['H', 'T'] for k in ['much worse', 'rest']}
print(f'Trump 2020 voters switching to Harris: recall {mirror(X, pub_piT):.1%}, corrected {mirror(X, piT):.1%}')

Trump 2020 voters switching to Harris: recall 2.3%, corrected 2.3%


## Reading

See technical appendix E.4a. The misclassified recalled switchers are spread across economy ratings about evenly (79% of recalled Biden→Trump voters rating the economy "much worse" had reported a Biden vote in 2020, 80% of the rest), so the correction lowers switch rates by roughly a fifth throughout without flattening the gradient.